# Extract Leverage Features (Amberdata)

This script extracts the following fields:
- open_interest_usd
- oi_momentum_5m
- oi_pct_change_5m
- oi_zscore_5m
- oi_momentum_15m
- oi_pct_change_15m
- oi_zscore_15m
- Funding_rate
- Cumulative Funding
- long_account
- short_account
- long_short_ratio
- long_short_skew
- basis_momentum_15m
- basis_pct
- mark_index_dislocation_momentum_15m
- mark_index_dislocation_pct
- liquidation_volume
- liquidation_acceleration_5m
- liquidation_acceleration_15m

An extract is created per Cryptocurrency coin:
- LTCUSDT
- BTCUSDT
- ETHUSDT
- SOLUSDT
- XRPUSDT
- DOGEUSDT
- BNBUSDT
- ADAUSDT
- LINKUSDT
- AVAXUSDT
- DOTUSDT
- BCHUSDT

In [ ]:
import os
import time
import requests
import pandas as pd
import numpy as np
from datetime import datetime, timedelta, UTC
from pathlib import Path

### OI Features

In [4]:

# Configuration

API_KEY = "" # insert key here
BASE_URL = ("https://api.amberdata.com/"
            "markets/futures/open-interest")
headers = {"x-api-key": API_KEY,"Accept": "application/json","Accept-Encoding": "gzip, deflate, br"}
exchange = "binance"
instruments = ["LTCUSDT",
               "BTCUSDT",
               "ETHUSDT",
               "SOLUSDT",
               "XRPUSDT",
               "DOGEUSDT",
               "BNBUSDT",
               "ADAUSDT",
               "LINKUSDT",
               "AVAXUSDT",
               "DOTUSDT",
               "BCHUSDT",]
BACKFILL_START_DATE = pd.Timestamp("2025-07-01",tz="UTC")
END_DATE = pd.Timestamp.utcnow()
CHUNK_DAYS = 7
MAX_RETRIES = 5
BASE_RETRY_WAIT = 10

# Functions
def get_output_files(instrument):
    symbol = instrument.lower()

    parquet_file = f"{symbol}_oi_features.parquet"
    csv_file = f"{symbol}_oi_features.csv"
    checkpoint_file = f"{symbol}_oi_checkpoint.parquet"
    return parquet_file, csv_file, checkpoint_file

def load_existing_data(parquet_file):
    if not os.path.exists(parquet_file):
        return pd.DataFrame()

    existing_df = pd.read_parquet(parquet_file)
    if existing_df.empty:
        return pd.DataFrame()
    existing_df["timestamp"] = pd.to_datetime(existing_df["timestamp"],utc=True)
    existing_df = (existing_df.drop_duplicates(subset=["timestamp"]).sort_values("timestamp").reset_index(drop=True))
    return existing_df

def determine_start_date(existing_df):
    if existing_df.empty:
        return BACKFILL_START_DATE
    last_timestamp = existing_df["timestamp"].max()
    incremental_start = last_timestamp)
    return incremental_start

def fetch_oi_chunk(instrument,current_start,current_end):
    url = f"{BASE_URL}/{instrument}"
    params = {"exchange": exchange,"startDate": current_start.strftime("%Y-%m-%dT%H:%M:%SZ"),"endDate": current_end.strftime("%Y-%m-%dT%H:%M:%SZ"),"timeFormat": "iso8601"}
    retries = 0
    while retries < MAX_RETRIES:
        try:
            response = requests.get(url,headers=headers,params=params,timeout=60)
            response.raise_for_status()
            data = response.json()
            rows = (data.get("payload", {}).get("data", []))
            print(f"Rows returned: {len(rows):,}")

            if not rows:
                return pd.DataFrame()

            chunk_df = pd.DataFrame([{"timestamp": row.get("exchangeTimestamp"),"open_interest": row.get("value")}for row in rows])
            chunk_df["timestamp"] = pd.to_datetime(chunk_df["timestamp"],utc=True)
            chunk_df["open_interest"] = pd.to_numeric(chunk_df["open_interest"],errors="coerce")
            chunk_df = (chunk_df.drop_duplicates(subset=["timestamp"]).sort_values("timestamp").reset_index(drop=True))
            return chunk_df

        except Exception as e:
            retries += 1
            wait_time = retries * BASE_RETRY_WAIT
            print(f"{instrument} retry {retries} "f"for {current_start} → {current_end}")
            print(e)
    return pd.DataFrame()

def pull_new_oi_data(instrument,current_start,end_date):
    all_chunks = []
    while current_start < end_date:
        current_end = min(current_start + timedelta(days=CHUNK_DAYS),end_date)
        print(f"\n{instrument} chunk: "f"{current_start} → {current_end}")
        chunk_df = fetch_oi_chunk(instrument=instrument,current_start=current_start,current_end=current_end)
        if not chunk_df.empty:
            all_chunks.append(chunk_df)
        current_start = current_end

    if not all_chunks:
        return pd.DataFrame()

    new_df = pd.concat(all_chunks,ignore_index=True)
    new_df = (new_df.drop_duplicates(subset=["timestamp"]).sort_values("timestamp").reset_index(drop=True))
    return new_df

def calculate_oi_features(oi_df,instrument):
    if oi_df.empty:
        return oi_df

    oi_df = oi_df.copy()
    oi_df["timestamp"] = pd.to_datetime(oi_df["timestamp"],utc=True)
    oi_df["open_interest"] = pd.to_numeric(oi_df["open_interest"],errors="coerce")
    oi_df = (oi_df.drop_duplicates(subset=["timestamp"], keep="last").sort_values("timestamp").reset_index(drop=True))

    # RESAMPLE TO 5M
    oi_df = (oi_df.set_index("timestamp").resample("5min").last().reset_index())
    oi_df["open_interest"] = oi_df["open_interest"].ffill()

    # FEATURE ENGINEERING
    oi_df["oi_momentum_5m"] = (oi_df["open_interest"].diff(1))
    oi_df["oi_momentum_15m"] = (oi_df["open_interest"].diff(3))
    oi_df["oi_pct_change_5m"] = (oi_df["open_interest"].pct_change(1))
    oi_df["oi_pct_change_15m"] = (oi_df["open_interest"].pct_change(3))
    rolling_1h = (oi_df["open_interest"].rolling(12))
    rolling_3h = (oi_df["open_interest"].rolling(36))
    oi_df["oi_zscore_1h"] = ((oi_df["open_interest"]- rolling_1h.mean())/rolling_1h.std())
    oi_df["oi_zscore_3h"] = ((oi_df["open_interest"] - rolling_3h.mean())/rolling_3h.std())
    oi_df["oi_zscore_5m"] = oi_df["oi_zscore_1h"]
    oi_df["oi_zscore_15m"] = oi_df["oi_zscore_3h"]
    oi_df["instrument"] = instrument
    oi_df = oi_df[["timestamp",
                   "instrument",
                   "open_interest",
                   "oi_momentum_5m",
                   "oi_momentum_15m",
                   "oi_pct_change_5m",
                   "oi_pct_change_15m",
                   "oi_zscore_5m",
                   "oi_zscore_15m",
                   "oi_zscore_1h",
                   "oi_zscore_3h"]]
    return oi_df

# OI EXTRACTION
def extract_oi_features(instrument):
    parquet_file, csv_file, checkpoint_file = get_output_files(instrument)
    print("\n" + "#" * 70)
    print(f"Starting OI extraction for {instrument}")
    print("#" * 70)

    existing_df = load_existing_data(parquet_file)
    current_start = determine_start_date(existing_df)
    print(f"Existing rows: {len(existing_df):,}")

    if not existing_df.empty:
        print(f"Existing latest timestamp: "f"{existing_df['timestamp'].max()}")

    print(f"Incremental start: {current_start}")
    print(f"End date: {END_DATE}")

    if current_start >= END_DATE:
        print(f"{instrument} is already up to date.")
        return

    new_df = pull_new_oi_data(
        instrument=instrument,
        current_start=current_start,
        end_date=END_DATE
    )

    if new_df.empty and existing_df.empty:
        print(f"No OI data for {instrument}. Skipping save.")
        return

    if new_df.empty:
        print("\nNo new rows fetched.")
        combined_df = existing_df.copy()
    else:
        print(f"\nNew rows before merge: {len(new_df):,}")
        combined_df = pd.concat([existing_df, new_df],ignore_index=True)

    final_df = calculate_oi_features(oi_df=combined_df,instrument=instrument)

    if final_df.empty:
        print(f"No final OI data for {instrument}. Skipping save.")
        return

    # Save outputs
    final_df.to_parquet(checkpoint_file,index=False)
    final_df.to_parquet(parquet_file,index=False)
    final_df.to_csv(csv_file,index=False)
    print(f"\nFinal shape {instrument}: {final_df.shape}")
    print(f"Final earliest timestamp: {final_df['timestamp'].min()}")
    print(f"Final latest timestamp: {final_df['timestamp'].max()}")
    print(final_df.head())
    print(f"Saved parquet: {parquet_file}")
    print(f"Saved csv: {csv_file}")
    print(f"Saved checkpoint: {checkpoint_file}")
    print(f"Finished {instrument}")

# Run script
for instrument in instruments:
    extract_oi_features(instrument)
    time.sleep(3)

print("\nOI extraction complete.")


######################################################################
Starting OI extraction for LTCUSDT
######################################################################
Existing rows: 95,752
Existing latest timestamp: 2026-05-29 11:15:00+00:00
Incremental start: 2026-05-29 11:00:00+00:00
End date: 2026-06-12 20:39:17.434858+00:00

LTCUSDT chunk: 2026-05-29 11:00:00+00:00 → 2026-06-05 11:00:00+00:00
Rows returned: 5,000

LTCUSDT chunk: 2026-06-05 11:00:00+00:00 → 2026-06-12 11:00:00+00:00
Rows returned: 5,000

LTCUSDT chunk: 2026-06-12 11:00:00+00:00 → 2026-06-12 20:39:17.434858+00:00
Rows returned: 579

New rows before merge: 10,579

Final shape LTCUSDT: (99896, 11)
Final earliest timestamp: 2025-07-01 00:00:00+00:00
Final latest timestamp: 2026-06-12 20:35:00+00:00
                  timestamp instrument  open_interest  oi_momentum_5m  \
0 2025-07-01 00:00:00+00:00    LTCUSDT    1590105.299             NaN   
1 2025-07-01 00:05:00+00:00    LTCUSDT    1590601.959         496.66

### Funding Features

In [5]:

# Configuration
API_KEY = "" # insert key here
BASE_URL = "https://api.amberdata.com/markets/futures/funding-rates"
exchange = "binance"
instruments = ["LTCUSDT",
               "BTCUSDT",
               "ETHUSDT",
               "SOLUSDT",
               "XRPUSDT",
               "DOGEUSDT",
               "BNBUSDT",
               "ADAUSDT",
               "LINKUSDT",
               "AVAXUSDT",
               "DOTUSDT",
               "BCHUSDT",]

headers = {"x-api-key": API_KEY,"Accept": "application/json","Accept-Encoding": "gzip, deflate, br",}

BACKFILL_START_DATE = datetime(2025, 7, 1,tzinfo=UTC)
END_DATE = datetime.now(UTC)
CHUNK_DAYS = 30
MAX_RETRIES = 5
BASE_RETRY_WAIT = 10
output_dir = Path.cwd()

# Functions
def get_symbol_name(instrument):
    return instrument.replace("USDT", "").lower()

def get_output_files(instrument):
    symbol = get_symbol_name(instrument)

    final_parquet_file = output_dir / f"{symbol}_funding.parquet"
    final_csv_file = output_dir / f"{symbol}_funding.csv"
    checkpoint_file = output_dir / f"{symbol}_funding_checkpoint.parquet"
    return final_parquet_file, final_csv_file, checkpoint_file

def load_existing_data(final_parquet_file, checkpoint_file):
    if final_parquet_file.exists():
        existing_df = pd.read_parquet(final_parquet_file)
    elif checkpoint_file.exists():
        existing_df = pd.read_parquet(checkpoint_file)
    else:
        return pd.DataFrame()
    if existing_df.empty:
        return pd.DataFrame()

    existing_df["timestamp"] = pd.to_datetime(existing_df["timestamp"],utc=True,errors="coerce")
    existing_df["funding_rate"] = pd.to_numeric(existing_df["funding_rate"],errors="coerce")
    existing_df = (existing_df.dropna(subset=["timestamp", "funding_rate"]).drop_duplicates(subset=["timestamp"], keep="last").sort_values("timestamp").reset_index(drop=True))
    return existing_df

def determine_start_date(existing_df):
    if existing_df.empty:
        return BACKFILL_START_DATE
    last_timestamp = existing_df["timestamp"].max()
    return last_timestamp

def fetch_funding_chunk(instrument,current_start,current_end):
    url = f"{BASE_URL}/{instrument}"
    params = {"exchange": exchange,"startDate": current_start.strftime("%Y-%m-%dT%H:%M:%SZ"),"endDate": current_end.strftime("%Y-%m-%dT%H:%M:%SZ"),"timeFormat": "iso8601","rateType": "applied",}
    retries = 0
    while retries < MAX_RETRIES:
        try:
            response = requests.get(url,headers=headers,params=params,timeout=60)

            if response.status_code == 400:
                print("400 Bad Request")
                print("URL:", response.url)
                print("Response:", response.text)
                return pd.DataFrame()

            if response.status_code == 401:
                print("401 Unauthorized")
                print("Response:", response.text)
                return pd.DataFrame()

            if response.status_code == 403:
                print("403 Forbidden")
                print("Response:", response.text)
                return pd.DataFrame()

            if response.status_code == 404:
                print("404 Not Found")
                print("URL:", response.url)
                print("Response:", response.text)
                return pd.DataFrame()

            if response.status_code == 410:
                print("410 Gone")
                print("URL:", response.url)
                print("Response:", response.text)
                return pd.DataFrame()

            response.raise_for_status()

            data = response.json()
            rows = (data.get("payload", {}).get("data", []))
            print(f"Returned {len(rows):,} rows")

            if not rows:
                return pd.DataFrame()

            parsed_rows = []

            for row in rows:
                row_instrument = row.get("instrument")
                if row_instrument is not None and row_instrument != instrument:
                    continue

                parsed_rows.append({"instrument": instrument,"exchange": exchange,"timestamp": row.get("exchangeTimestamp"),"funding_rate": row.get("fundingRate"),})

            chunk_df = pd.DataFrame(parsed_rows)

            if chunk_df.empty:
                return pd.DataFrame()

            chunk_df["timestamp"] = pd.to_datetime(chunk_df["timestamp"],utc=True,errors="coerce")
            chunk_df["funding_rate"] = pd.to_numeric(chunk_df["funding_rate"],errors="coerce")
            chunk_df = (chunk_df.dropna(subset=["timestamp", "funding_rate"]).drop_duplicates(subset=["timestamp"], keep="last").sort_values("timestamp").reset_index(drop=True))
            return chunk_df

        except requests.exceptions.RequestException as e:
            retries += 1
            print(f"{instrument} retry {retries}/{MAX_RETRIES} "f"for {current_start} → {current_end}")
            print(e)
            print(f"Waiting {wait_time}s")
    return pd.DataFrame()

def pull_new_funding_data(instrument,current_start,end_date):
    all_chunks = []

    while current_start < end_date:
        current_end = min(current_start + timedelta(days=CHUNK_DAYS),end_date)
        print(f"\n{instrument} chunk: "f"{current_start} → {current_end}")
        chunk_df = fetch_funding_chunk(instrument=instrument,current_start=current_start,current_end=current_end)

        if not chunk_df.empty:
            all_chunks.append(chunk_df)

        current_start = current_end
        time.sleep(1)

    if not all_chunks:
        return pd.DataFrame()

    new_df = pd.concat(all_chunks,ignore_index=True)
    new_df = (new_df.drop_duplicates(subset=["timestamp"], keep="last").sort_values("timestamp").reset_index(drop=True))
    return new_df


def finalize_funding_data(
    funding_df,
    instrument
):
    if funding_df.empty:
        return funding_df

    funding_df = funding_df.copy()
    funding_df["timestamp"] = pd.to_datetime(funding_df["timestamp"],utc=True,errors="coerce")
    funding_df["funding_rate"] = pd.to_numeric(funding_df["funding_rate"],errors="coerce")
    funding_df["instrument"] = instrument
    funding_df["exchange"] = exchange
    funding_df = (funding_df.dropna(subset=["timestamp", "funding_rate"]).drop_duplicates(subset=["timestamp"], keep="last").sort_values("timestamp").reset_index(drop=True))
    funding_df["cumulative_funding"] = (funding_df["funding_rate"].cumsum())
    funding_df = funding_df[["timestamp","instrument","exchange","funding_rate","cumulative_funding"]]
    return funding_df

# extraction
def extract_funding(instrument):
    final_parquet_file, final_csv_file, checkpoint_file = get_output_files(instrument)
    print("\n" + "#" * 70)
    print(f"Starting funding extraction for {instrument}")
    print("#" * 70)

    existing_df = load_existing_data(final_parquet_file=final_parquet_file,checkpoint_file=checkpoint_file)
    current_start = determine_start_date(existing_df)
    print(f"Existing rows: {len(existing_df):,}")
    if not existing_df.empty:
        print(f"Existing latest timestamp: "f"{existing_df['timestamp'].max()}")
    print(f"Incremental start: {current_start}")
    print(f"End date: {END_DATE}")

    if current_start >= END_DATE:
        print(f"{instrument} is already up to date.")
        return existing_df

    new_df = pull_new_funding_data(instrument=instrument,current_start=current_start,end_date=END_DATE)

    if new_df.empty and existing_df.empty:
        print(f"No funding data for {instrument}. Skipping save.")
        return None

    if new_df.empty:
        print("\nNo new rows fetched.")
        combined_df = existing_df.copy()
    else:
        print(f"\nNew rows before merge: {len(new_df):,}")
        combined_df = pd.concat([existing_df, new_df],ignore_index=True)

    funding_df = finalize_funding_data(funding_df=combined_df,instrument=instrument)
    if funding_df.empty:
        print(f"No valid funding rows for {instrument}. Skipping save.")
        return None

    funding_df.to_parquet(checkpoint_file,index=False)
    funding_df.to_parquet(final_parquet_file,index=False)
    funding_df.to_csv(final_csv_file,index=False)
    print(f"\nFinal shape {instrument}: {funding_df.shape}")
    print(f"Final earliest timestamp: {funding_df['timestamp'].min()}")
    print(f"Final latest timestamp: {funding_df['timestamp'].max()}")
    print(funding_df.head())
    print(f"Saved parquet: {final_parquet_file}")
    print(f"Saved csv: {final_csv_file}")
    print(f"Saved checkpoint: {checkpoint_file}")
    print(f"Finished {instrument}")
    return funding_df

# Run script
results = {}
for instrument in instruments:
    df = extract_funding(instrument)
    if df is not None:
        results[instrument] = df
print("\nAll funding extraction complete.")
print("Successful instruments:", list(results.keys()))


######################################################################
Starting funding extraction for LTCUSDT
######################################################################
Existing rows: 995
Existing latest timestamp: 2026-06-05 16:00:00.009000+00:00
Incremental start: 2026-06-04 16:00:00.009000+00:00
End date: 2026-06-12 20:41:44.217323+00:00

LTCUSDT chunk: 2026-06-04 16:00:00.009000+00:00 → 2026-06-12 20:41:44.217323+00:00
Returned 25 rows

New rows before merge: 25

Final shape LTCUSDT: (1016, 5)
Final earliest timestamp: 2025-07-01 00:00:00.007000+00:00
Final latest timestamp: 2026-06-12 16:00:00+00:00
                         timestamp instrument exchange  funding_rate  \
0 2025-07-01 00:00:00.007000+00:00    LTCUSDT  binance      0.000071   
1 2025-07-01 08:00:00.002000+00:00    LTCUSDT  binance      0.000100   
2        2025-07-01 16:00:00+00:00    LTCUSDT  binance      0.000009   
3 2025-07-02 00:00:00.001000+00:00    LTCUSDT  binance     -0.000096   
4        2025-

### long_short_skew

In [6]:

# Configuration
API_KEY = "" # insert key here
BASE_URL = ("https://api.amberdata.com/"
            "markets/futures/long-short-ratio")
exchange = "binance"
instruments = ["LTCUSDT",
               "BTCUSDT",
               "ETHUSDT",
               "SOLUSDT",
               "XRPUSDT",
               "DOGEUSDT",
               "BNBUSDT",
               "ADAUSDT",
               "LINKUSDT",
               "AVAXUSDT",
               "DOTUSDT",
               "BCHUSDT",]

headers = {"x-api-key": API_KEY,"Accept": "application/json","Accept-Encoding": "gzip, deflate, br"}
BACKFILL_START_DATE = datetime(2025, 7, 1,tzinfo=UTC)
END_DATE = datetime.now(UTC)
CHUNK_DAYS = 30
MAX_RETRIES = 5
BASE_RETRY_WAIT = 10
output_dir = Path.cwd()

# Functions
def get_symbol_name(instrument):
    return instrument.replace("USDT", "").lower()

def get_output_files(instrument):
    symbol = get_symbol_name(instrument)
    final_parquet_file = output_dir / f"{symbol}_longshort.parquet"
    final_csv_file = output_dir / f"{symbol}_longshort.csv"
    checkpoint_file = output_dir / f"{symbol}_longshort_checkpoint.parquet"
    return final_parquet_file, final_csv_file, checkpoint_file

def load_existing_data(final_parquet_file, checkpoint_file):
    if final_parquet_file.exists():
        existing_df = pd.read_parquet(final_parquet_file)
    elif checkpoint_file.exists():
        existing_df = pd.read_parquet(checkpoint_file)
    else:
        return pd.DataFrame()

    if existing_df.empty:
        return pd.DataFrame()

    existing_df["timestamp"] = pd.to_datetime(existing_df["timestamp"],utc=True,errors="coerce")
    numeric_columns = ["long_account","short_account","long_short_ratio","long_short_skew"]
    for col in numeric_columns:
        if col in existing_df.columns:
            existing_df[col] = pd.to_numeric(existing_df[col],errors="coerce")

    existing_df = (existing_df.dropna(subset=["timestamp"]).drop_duplicates(subset=["timestamp"], keep="last").sort_values("timestamp").reset_index(drop=True))
    return existing_df

def determine_start_date(existing_df):
    if existing_df.empty:
        return BACKFILL_START_DATE
    last_timestamp = existing_df["timestamp"].max()
    return last_timestamp

def fetch_long_short_chunk(instrument,current_start,current_end):
    url = f"{BASE_URL}/{instrument}"
    params = {"exchange": exchange,"startDate": current_start.strftime("%Y-%m-%dT%H:%M:%SZ"),"endDate": current_end.strftime("%Y-%m-%dT%H:%M:%SZ"),"timeFormat": "iso8601","metricType": "default"}
    retries = 0
    while retries < MAX_RETRIES:
        try:
            response = requests.get(url,headers=headers,params=params,timeout=60)

            if response.status_code == 400:
                print("400 Bad Request")
                print("URL:", response.url)
                print("Response:", response.text)
                return pd.DataFrame()

            if response.status_code == 401:
                print("401 Unauthorized")
                print("Response:", response.text)
                return pd.DataFrame()

            if response.status_code == 403:
                print("403 Forbidden")
                print("Response:", response.text)
                return pd.DataFrame()

            if response.status_code == 404:
                print("404 Not Found")
                print("URL:", response.url)
                print("Response:", response.text)
                return pd.DataFrame()

            if response.status_code == 410:
                print("410 Gone")
                print("URL:", response.url)
                print("Response:", response.text)
                return pd.DataFrame()

            response.raise_for_status()
            data = response.json()

            rows = (data.get("payload", {}).get("data", []))
            print(f"Returned {len(rows):,} rows")

            if not rows:
                return pd.DataFrame()

            parsed_rows = []
            for row in rows:
                long_account = row.get("longAccount")
                short_account = row.get("shortAccount")
                ratio = row.get("ratio")
                if long_account is None or short_account is None:
                    continue

                parsed_rows.append({ "instrument": instrument,"exchange": exchange,"timestamp": row.get("exchangeTimestamp"), "long_account": long_account,"short_account": short_account,"long_short_ratio": ratio,})

            chunk_df = pd.DataFrame(parsed_rows)

            if chunk_df.empty:
                return pd.DataFrame()

            chunk_df["timestamp"] = pd.to_datetime(chunk_df["timestamp"],utc=True,errors="coerce")
            numeric_columns = ["long_account","short_account","long_short_ratio"]

            for col in numeric_columns:
                chunk_df[col] = pd.to_numeric(chunk_df[col],errors="coerce")

            chunk_df["long_short_skew"] = ( chunk_df["long_account"] - chunk_df["short_account"])
            chunk_df = (chunk_df.dropna(subset=["timestamp", "long_account", "short_account"]).drop_duplicates(subset=["timestamp"], keep="last").sort_values("timestamp").reset_index(drop=True))

            return chunk_df

        except requests.exceptions.RequestException as e:
            retries += 1
            print(f"{instrument} retry {retries}/{MAX_RETRIES} "f"for {current_start} → {current_end}")
            print(e)
    return pd.DataFrame()

def pull_new_long_short_data(instrument,current_start,end_date):
    all_chunks = []

    while current_start < end_date:
        current_end = min(current_start + timedelta(days=CHUNK_DAYS),end_date)
        print(f"\n{instrument} chunk: "f"{current_start} → {current_end}")
        chunk_df = fetch_long_short_chunk(instrument=instrument,current_start=current_start,current_end=current_end)
        if not chunk_df.empty:
            all_chunks.append(chunk_df)
        current_start = current_end
    if not all_chunks:
        return pd.DataFrame()

    new_df = pd.concat(all_chunks,ignore_index=True)
    new_df = (new_df.drop_duplicates(subset=["timestamp"], keep="last").sort_values("timestamp").reset_index(drop=True))
    return new_df

def finalize_long_short_data(ls_df,instrument):
    if ls_df.empty:
        return ls_df

    ls_df = ls_df.copy()
    ls_df["timestamp"] = pd.to_datetime(ls_df["timestamp"],utc=True,errors="coerce")
    ls_df["instrument"] = instrument
    ls_df["exchange"] = exchange

    numeric_columns = ["long_account","short_account","long_short_ratio"]

    for col in numeric_columns:
        ls_df[col] = pd.to_numeric(ls_df[col],errors="coerce")
    ls_df["long_short_skew"] = (ls_df["long_account"]-ls_df["short_account"])
    ls_df = (ls_df.dropna(subset=["timestamp", "long_account", "short_account"]).drop_duplicates(subset=["timestamp"], keep="last").sort_values("timestamp").reset_index(drop=True))
    ls_df = ls_df[["timestamp","instrument","exchange","long_account","short_account","long_short_ratio","long_short_skew"]]
    return ls_df

# extraction
def extract_long_short(instrument):
    final_parquet_file, final_csv_file, checkpoint_file = get_output_files(instrument)
    print("\n" + "#" * 70)
    print(f"Starting long-short extraction for {instrument}")
    print("#" * 70)
    existing_df = load_existing_data(final_parquet_file=final_parquet_file,checkpoint_file=checkpoint_file)
    current_start = determine_start_date(existing_df)
    print(f"Existing rows: {len(existing_df):,}")
    
    if not existing_df.empty:
        print(f"Existing latest timestamp: "f"{existing_df['timestamp'].max()}")
        
    print(f"Incremental start: {current_start}")
    print(f"End date: {END_DATE}")
    
    if current_start >= END_DATE:
        print(f"{instrument} is already up to date.")
        return existing_df

    new_df = pull_new_long_short_data(instrument=instrument,current_start=current_start,end_date=END_DATE)

    if new_df.empty and existing_df.empty:
        print(f"No long-short data for {instrument}. Skipping save.")
        return None

    if new_df.empty:
        print("\nNo new rows fetched.")
        combined_df = existing_df.copy()
    else:
        print(f"\nNew rows before merge: {len(new_df):,}")
        combined_df = pd.concat([existing_df, new_df],ignore_index=True)

    ls_df = finalize_long_short_data(ls_df=combined_df,instrument=instrument)

    if ls_df.empty:
        print(f"No valid long-short rows for {instrument}. Skipping save.")
        return None

    # save results
    ls_df.to_parquet(checkpoint_file,index=False)
    ls_df.to_parquet(final_parquet_file,index=False)
    ls_df.to_csv(final_csv_file,index=False)
    print(f"\nFinal shape {instrument}: {ls_df.shape}")
    print(f"Final earliest timestamp: {ls_df['timestamp'].min()}")
    print(f"Final latest timestamp: {ls_df['timestamp'].max()}")
    print(ls_df.head())
    print(f"Saved parquet: {final_parquet_file}")
    print(f"Saved csv: {final_csv_file}")
    print(f"Saved checkpoint: {checkpoint_file}")
    print(f"Finished {instrument}")
    return ls_df

# run script
results = {}

for instrument in instruments:
    df = extract_long_short(instrument)
    if df is not None:
        results[instrument] = df

print("\nAll long-short extraction complete.")
print("Successful instruments:", list(results.keys()))


######################################################################
Starting long-short extraction for LTCUSDT
######################################################################
Existing rows: 8,143
Existing latest timestamp: 2026-06-05 07:00:00+00:00
Incremental start: 2026-06-04 07:00:00+00:00
End date: 2026-06-12 20:43:00.961911+00:00

LTCUSDT chunk: 2026-06-04 07:00:00+00:00 → 2026-06-12 20:43:00.961911+00:00
Returned 206 rows

New rows before merge: 206

Final shape LTCUSDT: (8324, 7)
Final earliest timestamp: 2025-07-01 00:00:00+00:00
Final latest timestamp: 2026-06-12 20:00:00+00:00
                  timestamp instrument exchange  long_account  short_account  \
0 2025-07-01 00:00:00+00:00    LTCUSDT  binance        0.6790         0.3210   
1 2025-07-01 01:00:00+00:00    LTCUSDT  binance        0.6786         0.3214   
2 2025-07-01 02:00:00+00:00    LTCUSDT  binance        0.6660         0.3340   
3 2025-07-01 03:00:00+00:00    LTCUSDT  binance        0.6623         0.337

### Basis and Mark-Index Dislocation

In [7]:

# Configuration
API_KEY = "" # insert key here
BASE_URL = ("https://api.amberdata.com/"
            "markets/futures/tickers")
exchange = "binance"
instruments = ["LTCUSDT",
               "BTCUSDT",
               "ETHUSDT",
               "SOLUSDT",
               "XRPUSDT",
               "DOGEUSDT",
               "BNBUSDT",
               "ADAUSDT",
               "LINKUSDT",
               "AVAXUSDT",
               "DOTUSDT",
               "BCHUSDT",]
headers = {"x-api-key": API_KEY,"Accept": "application/json","Accept-Encoding": "gzip, deflate, br"}
BACKFILL_START_DATE = datetime(2025, 7, 1,tzinfo=UTC)
END_DATE = datetime.now(UTC)
CHUNK_DAYS = 7
MAX_RETRIES = 5
BASE_RETRY_WAIT = 10
output_dir = Path.cwd()

# Functions
def get_symbol_name(instrument):
    return instrument.replace("USDT", "").lower()

def get_output_files(instrument):
    symbol = get_symbol_name(instrument)
    final_parquet_file = output_dir / f"{symbol}_basis.parquet"
    final_csv_file = output_dir / f"{symbol}_basis.csv"
    checkpoint_file = output_dir / f"{symbol}_basis_checkpoint.parquet"
    return final_parquet_file, final_csv_file, checkpoint_file

def load_existing_data(final_parquet_file, checkpoint_file):
    if final_parquet_file.exists():
        existing_df = pd.read_parquet(final_parquet_file)
    elif checkpoint_file.exists():
        existing_df = pd.read_parquet(checkpoint_file)
    else:
        return pd.DataFrame()

    if existing_df.empty:
        return pd.DataFrame()

    existing_df["timestamp"] = pd.to_datetime(existing_df["timestamp"],utc=True,errors="coerce")
    numeric_columns = ["basis_pct","mark_index_dislocation_pct","basis_momentum_15m","mark_index_dislocation_momentum_15m"]

    for col in numeric_columns:
        if col in existing_df.columns:
            existing_df[col] = pd.to_numeric(existing_df[col],errors="coerce")

    existing_df = (existing_df.dropna(subset=["timestamp"]).drop_duplicates(subset=["timestamp"], keep="last").sort_values("timestamp").reset_index(drop=True))
    return existing_df

def determine_start_date(existing_df):
    if existing_df.empty:
        return BACKFILL_START_DATE
    last_timestamp = existing_df["timestamp"].max()
    return last_timestamp

def fetch_basis_chunk(
    instrument,
    current_start,
    current_end
):
    url = f"{BASE_URL}/{instrument}"
    params = {"exchange": exchange,"startDate": current_start.strftime("%Y-%m-%dT%H:%M:%SZ"),"endDate": current_end.strftime("%Y-%m-%dT%H:%M:%SZ"),"timeFormat": "iso8601"}
    retries = 0

    while retries < MAX_RETRIES:
        try:
            response = requests.get(url,headers=headers,params=params,timeout=60)

            if response.status_code == 400:
                print("400 Bad Request")
                print("URL:", response.url)
                print("Response:", response.text)
                return pd.DataFrame()

            if response.status_code == 401:
                print("401 Unauthorized")
                print("Response:", response.text)
                return pd.DataFrame()

            if response.status_code == 403:
                print("403 Forbidden")
                print("Response:", response.text)
                return pd.DataFrame()

            if response.status_code == 404:
                print("404 Not Found")
                print("URL:", response.url)
                print("Response:", response.text)
                return pd.DataFrame()

            if response.status_code == 410:
                print("410 Gone")
                print("URL:", response.url)
                print("Response:", response.text)
                return pd.DataFrame()

            response.raise_for_status()
            data = response.json()
            rows = (data.get("payload", {}).get("data", []))
            print(f"Returned {len(rows):,} rows")

            if not rows:
                return pd.DataFrame()

            parsed_rows = []

            for row in rows:
                bid = row.get("bid")
                ask = row.get("ask")
                mark = row.get("markPrice")
                index = row.get("indexPrice")

                if bid is None or ask is None or mark is None or index is None:
                    continue

                bid = pd.to_numeric(bid, errors="coerce")
                ask = pd.to_numeric(ask, errors="coerce")
                mark = pd.to_numeric(mark, errors="coerce")
                index = pd.to_numeric(index, errors="coerce")

                if pd.isna(bid) or pd.isna(ask) or pd.isna(mark) or pd.isna(index):
                    continue

                if index == 0:
                    continue

                mid = (bid + ask) / 2
                basis_pct = ( mid - index) / index
                dislocation_pct = (mark - index) / index
                parsed_rows.append({"instrument": instrument,"exchange": exchange,"timestamp": row.get("exchangeTimestamp"),"basis_pct": basis_pct,"mark_index_dislocation_pct": dislocation_pct})
            chunk_df = pd.DataFrame(parsed_rows)

            if chunk_df.empty:
                return pd.DataFrame()

            chunk_df["timestamp"] = pd.to_datetime(chunk_df["timestamp"],utc=True,errors="coerce")
            chunk_df["basis_pct"] = pd.to_numeric(chunk_df["basis_pct"],errors="coerce")
            chunk_df["mark_index_dislocation_pct"] = pd.to_numeric(chunk_df["mark_index_dislocation_pct"],errors="coerce")
            chunk_df = (chunk_df.dropna(subset=["timestamp","basis_pct","mark_index_dislocation_pct"]).drop_duplicates(subset=["timestamp"], keep="last").sort_values("timestamp").reset_index(drop=True))
            return chunk_df

        except requests.exceptions.RequestException as e:
            retries += 1
            print(f"{instrument} retry {retries}/{MAX_RETRIES} "f"for {current_start} → {current_end}")
            print(e)
    return pd.DataFrame()

def pull_new_basis_data(instrument,current_start,end_date):
    all_chunks = []
    while current_start < end_date:
        current_end = min(current_start + timedelta(days=CHUNK_DAYS),end_date)
        print(f"\n{instrument} chunk: "f"{current_start} → {current_end}")
        chunk_df = fetch_basis_chunk(instrument=instrument,current_start=current_start,current_end=current_end)
        
        if not chunk_df.empty:
            all_chunks.append(chunk_df)
        current_start = current_end

    if not all_chunks:
        return pd.DataFrame()

    new_df = pd.concat(all_chunks,ignore_index=True)
    new_df = (new_df.drop_duplicates(subset=["timestamp"], keep="last").sort_values("timestamp").reset_index(drop=True))
    return new_df

def finalize_basis_data(basis_df,instrument):
    if basis_df.empty:
        return basis_df

    basis_df = basis_df.copy()
    basis_df["timestamp"] = pd.to_datetime(basis_df["timestamp"],utc=True,errors="coerce")
    basis_df["instrument"] = instrument
    basis_df["exchange"] = exchange
    numeric_columns = ["basis_pct","mark_index_dislocation_pct"]
    for col in numeric_columns:
        basis_df[col] = pd.to_numeric(basis_df[col],errors="coerce")

    basis_df = (basis_df.dropna(subset=["timestamp","basis_pct","mark_index_dislocation_pct"]).drop_duplicates(subset=["timestamp"], keep="last").sort_values("timestamp").reset_index(drop=True))
    basis_df["basis_momentum_15m"] = (basis_df["basis_pct"].diff(15))
    basis_df["mark_index_dislocation_momentum_15m"] = (basis_df["mark_index_dislocation_pct"].diff(15))
    basis_df = basis_df[["timestamp","instrument","exchange","basis_pct","mark_index_dislocation_pct","basis_momentum_15m","mark_index_dislocation_momentum_15m"]]
    return basis_df

def extract_basis_dislocation(instrument):
    final_parquet_file, final_csv_file, checkpoint_file = get_output_files(instrument)

    print("\n" + "#" * 70)
    print(f"Starting basis/dislocation for {instrument}")
    print("#" * 70)

    existing_df = load_existing_data(final_parquet_file=final_parquet_file,checkpoint_file=checkpoint_file)
    current_start = determine_start_date(existing_df)
    print(f"Existing rows: {len(existing_df):,}")

    if not existing_df.empty:
        print(f"Existing latest timestamp: " f"{existing_df['timestamp'].max()}")

    print(f"Incremental start: {current_start}")
    print(f"End date: {END_DATE}")

    if current_start >= END_DATE:
        print(f"{instrument} is already up to date.")
        return existing_df

    new_df = pull_new_basis_data(instrument=instrument,current_start=current_start,end_date=END_DATE)
    if new_df.empty and existing_df.empty:
        print(f"No basis/dislocation data for {instrument}. Skipping save.")
        return None

    if new_df.empty:
        print("\nNo new rows fetched.")
        combined_df = existing_df.copy()
    else:
        print(f"\nNew rows before merge: {len(new_df):,}")
        combined_df = pd.concat([existing_df, new_df],ignore_index=True)
        
    basis_df = finalize_basis_data(basis_df=combined_df,instrument=instrument)

    if basis_df.empty:
        print(f"No valid basis/dislocation rows for {instrument}. Skipping save.")
        return None

    # save results
    basis_df.to_parquet(checkpoint_file,index=False)
    basis_df.to_parquet(final_parquet_file,index=False)
    basis_df.to_csv(final_csv_file,index=False)
    print(f"\nFinal shape {instrument}: {basis_df.shape}")
    print(f"Final earliest timestamp: {basis_df['timestamp'].min()}")
    print(f"Final latest timestamp: {basis_df['timestamp'].max()}")
    print(basis_df.head())
    print(f"Saved parquet: {final_parquet_file}")
    print(f"Saved csv: {final_csv_file}")
    print(f"Saved checkpoint: {checkpoint_file}")
    print(f"Finished {instrument}")
    return basis_df

# run script
results = {}

for instrument in instruments:
    df = extract_basis_dislocation(instrument)
    if df is not None:
        results[instrument] = df

print("\nAll basis extraction complete.")
print("Successful instruments:", list(results.keys()))


######################################################################
Starting basis/dislocation for LTCUSDT
######################################################################
Existing rows: 750,606
Existing latest timestamp: 2026-06-02 00:23:39.960000+00:00
Incremental start: 2026-06-01 23:23:39.960000+00:00
End date: 2026-06-12 20:44:22.577652+00:00

LTCUSDT chunk: 2026-06-01 23:23:39.960000+00:00 → 2026-06-08 23:23:39.960000+00:00
Returned 22,625 rows

LTCUSDT chunk: 2026-06-08 23:23:39.960000+00:00 → 2026-06-12 20:44:22.577652+00:00
Returned 22,771 rows

New rows before merge: 33,490

Final shape LTCUSDT: (784096, 7)
Final earliest timestamp: 2025-07-01 00:00:00.016000+00:00
Final latest timestamp: 2026-06-08 23:41:23.328000+00:00
                         timestamp instrument exchange  basis_pct  \
0 2025-07-01 00:00:00.016000+00:00    LTCUSDT  binance  -0.000450   
1 2025-07-01 00:00:04.445000+00:00    LTCUSDT  binance  -0.000307   
2 2025-07-01 00:00:04.471000+00:00    LTCU

### Liquidation

In [8]:

# Configuration
API_KEY = "" # insert key here
BASE_URL = ("https://api.amberdata.com/"
            "markets/futures/liquidations")
exchange = "binance"
instruments = ["LTCUSDT",
               "BTCUSDT",
               "ETHUSDT",
               "SOLUSDT",
               "XRPUSDT",
               "DOGEUSDT",
               "BNBUSDT",
               "ADAUSDT",
               "LINKUSDT",
               "AVAXUSDT",
               "DOTUSDT",
               "BCHUSDT"]
headers = {"x-api-key": API_KEY,"Accept": "application/json","Accept-Encoding": "gzip, deflate, br"}
BACKFILL_START_DATE = datetime(2025, 7, 1,tzinfo=UTC)
END_DATE = datetime.now(UTC)
CHUNK_DAYS = 7
MAX_RETRIES = 5
BASE_RETRY_WAIT = 10
output_dir = Path.cwd()

# Functions
def get_symbol_name(instrument):
    return instrument.replace("USDT", "").lower()

def get_output_files(instrument):
    symbol = get_symbol_name(instrument)
    final_parquet_file = output_dir / f"{symbol}_liquidations.parquet"
    final_csv_file = output_dir / f"{symbol}_liquidations.csv"
    checkpoint_file = output_dir / f"{symbol}_liq_checkpoint.parquet"
    return final_parquet_file, final_csv_file, checkpoint_file

def load_existing_data(final_parquet_file, checkpoint_file):
    if final_parquet_file.exists():
        existing_df = pd.read_parquet(final_parquet_file)
    elif checkpoint_file.exists():
        existing_df = pd.read_parquet(checkpoint_file)
    else:
        return pd.DataFrame()
    if existing_df.empty:
        return pd.DataFrame()

    existing_df["timestamp"] = pd.to_datetime(existing_df["timestamp"],utc=True,errors="coerce")
    numeric_columns = ["price","volume","liquidation_volume","liquidation_acceleration_5m","liquidation_acceleration_15m"]
    for col in numeric_columns:
        if col in existing_df.columns:
            existing_df[col] = pd.to_numeric(existing_df[col],errors="coerce")

    existing_df = (existing_df.dropna(subset=["timestamp"]).drop_duplicates().sort_values("timestamp").reset_index(drop=True))
    return existing_df

def determine_start_date(existing_df):
    if existing_df.empty:
        return BACKFILL_START_DATE
    last_timestamp = existing_df["timestamp"].max()
    return last_timestamp

def fetch_liquidation_chunk(
    instrument,
    current_start,
    current_end
):
    url = f"{BASE_URL}/{instrument}"
    params = {"exchange": exchange,"startDate": current_start.strftime("%Y-%m-%dT%H:%M:%SZ"),"endDate": current_end.strftime("%Y-%m-%dT%H:%M:%SZ"),"timeFormat": "iso8601"}

    retries = 0

    while retries < MAX_RETRIES:
        try:
            response = requests.get(url,headers=headers,params=params,timeout=60)

            if response.status_code == 400:
                print("400 Bad Request")
                print("URL:", response.url)
                print("Response:", response.text)
                return pd.DataFrame()

            if response.status_code == 401:
                print("401 Unauthorized")
                print("Response:", response.text)
                return pd.DataFrame()

            if response.status_code == 403:
                print("403 Forbidden")
                print("Response:", response.text)
                return pd.DataFrame()

            if response.status_code == 404:
                print("404 Not Found")
                print("URL:", response.url)
                print("Response:", response.text)
                return pd.DataFrame()

            if response.status_code == 410:
                print("410 Gone")
                print("URL:", response.url)
                print("Response:", response.text)
                return pd.DataFrame()

            response.raise_for_status()
            data = response.json()
            rows = (data.get("payload", {}).get("data", []))

            print(f"Returned {len(rows):,} rows")

            if not rows:
                return pd.DataFrame()

            parsed_rows = []

            for row in rows:
                timestamp = row.get("exchangeTimestamp")
                price = pd.to_numeric(row.get("price"),errors="coerce")
                volume = pd.to_numeric(row.get("volume"),errors="coerce")

                if timestamp is None or pd.isna(price) or pd.isna(volume):
                    continue

                parsed_rows.append(
                    {
                        "instrument": row.get("instrument", instrument),
                        "exchange": row.get("exchange", exchange),
                        "timestamp": timestamp,
                        "price": price,
                        "volume": volume,
                        "unit": row.get("unit"),
                        "liquidation_volume": price * volume,
                        "side": row.get("side"),
                        "position_type": row.get("positionType"),
                        "status": row.get("status"),
                        "order_type": row.get("type")
                    }
                )

            chunk_df = pd.DataFrame(parsed_rows)

            if chunk_df.empty:
                return pd.DataFrame()

            chunk_df["timestamp"] = pd.to_datetime(chunk_df["timestamp"],utc=True,errors="coerce")
            numeric_columns = ["price","volume","liquidation_volume"]

            for col in numeric_columns:
                chunk_df[col] = pd.to_numeric(chunk_df[col],errors="coerce")

            chunk_df = (chunk_df.dropna(subset=["timestamp","liquidation_volume"]).drop_duplicates().sort_values("timestamp").reset_index(drop=True))
            print(f"Parsed {len(chunk_df):,} rows from this chunk")
            return chunk_df

        except requests.exceptions.RequestException as e:
            retries += 1
            print(f"{instrument} retry {retries}/{MAX_RETRIES} " f"for {current_start} → {current_end}")
            print(e)

    return pd.DataFrame()

def pull_new_liquidation_data(instrument,current_start,end_date):
    all_chunks = []

    while current_start < end_date:
        current_end = min(current_start + timedelta(days=CHUNK_DAYS),end_date)

        print(f"\n{instrument} chunk: "f"{current_start} → {current_end}")
        chunk_df = fetch_liquidation_chunk(instrument=instrument,current_start=current_start,current_end=current_end)

        if not chunk_df.empty:
            all_chunks.append(chunk_df)

        current_start = current_end

    if not all_chunks:
        return pd.DataFrame()

    new_df = pd.concat(all_chunks,ignore_index=True)
    new_df = (new_df.drop_duplicates().sort_values("timestamp").reset_index(drop=True))
    return new_df

def finalize_liquidation_data(
    liq_df,
    instrument
):
    if liq_df.empty:
        return liq_df

    liq_df = liq_df.copy()
    liq_df["timestamp"] = pd.to_datetime(liq_df["timestamp"],utc=True,errors="coerce")

    if "liquidation_volume" not in liq_df.columns:
        liq_df["price"] = pd.to_numeric(liq_df["price"],errors="coerce")
        liq_df["volume"] = pd.to_numeric(liq_df["volume"],errors="coerce")
        liq_df["liquidation_volume"] = (liq_df["price"] * liq_df["volume"])

    liq_df["liquidation_volume"] = pd.to_numeric(liq_df["liquidation_volume"],errors="coerce")
    liq_df = (liq_df.dropna(subset=["timestamp","liquidation_volume"]).drop_duplicates().sort_values("timestamp").reset_index(drop=True))
    if liq_df.empty:
        return liq_df

    # --------------------------------
    # AGGREGATE TO 1 MINUTE
    # --------------------------------
    # If the existing file is already aggregated, this still works because
    # liquidation_volume remains the field being summed by minute.

    liq_df = (liq_df.set_index("timestamp").groupby(pd.Grouper(freq="1min")).agg(liquidation_volume=("liquidation_volume", "sum")).reset_index())
    liq_df["liquidation_volume"] = (liq_df["liquidation_volume"].fillna(0))
    liq_df["instrument"] = instrument
    liq_df["exchange"] = exchange

    # --------------------------------
    # LIQUIDATION ACCELERATION
    # --------------------------------

    liq_df["liquidation_acceleration_5m"] = (liq_df["liquidation_volume"].diff(5))
    liq_df["liquidation_acceleration_15m"] = (liq_df["liquidation_volume"].diff(15))
    liq_df = liq_df[["timestamp","instrument","exchange","liquidation_volume","liquidation_acceleration_5m","liquidation_acceleration_15m"]]
    return liq_df

def extract_liquidations(instrument):
    final_parquet_file, final_csv_file, checkpoint_file = get_output_files(instrument)

    print("\n######################################################################")
    print(f"Starting liquidation extraction for {instrument}")
    print("######################################################################")

    existing_df = load_existing_data(final_parquet_file=final_parquet_file,checkpoint_file=checkpoint_file)

    current_start = determine_start_date(existing_df)

    print(f"Existing rows: {len(existing_df):,}")
    if not existing_df.empty:
        print(f"Existing latest timestamp: " f"{existing_df['timestamp'].max()}")

    print(f"Incremental start: {current_start}")
    print(f"End date: {END_DATE}")

    if current_start >= END_DATE:
        print(f"{instrument} is already up to date.")
        return existing_df

    new_df = pull_new_liquidation_data(instrument=instrument,current_start=current_start,end_date=END_DATE)

    if new_df.empty and existing_df.empty:
        print(f"No liquidation data for {instrument}. Skipping save.")
        return None

    if new_df.empty:
        print("\nNo new rows fetched.")
        combined_df = existing_df.copy()
    else:
        print(f"\nNew rows before merge: {len(new_df):,}")
        combined_df = pd.concat([existing_df, new_df],ignore_index=True)

    liq_df = finalize_liquidation_data(liq_df=combined_df,instrument=instrument)
    if liq_df.empty:
        print(f"No valid liquidation rows for {instrument}. Skipping save.")
        return None

    # save results
    liq_df.to_parquet(checkpoint_file,index=False)
    liq_df.to_parquet(final_parquet_file,index=False)
    liq_df.to_csv(final_csv_file,index=False)
    print(f"\nFinal shape {instrument}: {liq_df.shape}")
    print(f"Final earliest timestamp: {liq_df['timestamp'].min()}")
    print(f"Final latest timestamp: {liq_df['timestamp'].max()}")
    print(liq_df.head())
    print(f"Saved parquet: {final_parquet_file}")
    print(f"Saved csv: {final_csv_file}")
    print(f"Saved checkpoint: {checkpoint_file}")
    print(f"Finished {instrument}")
    return liq_df

# Run script
results = {}

for instrument in instruments:
    df = extract_liquidations(instrument)
    if df is not None:
        results[instrument] = df


print("\nAll liquidation extraction complete.")
print("Successful instruments:", list(results.keys()))


######################################################################
Starting liquidation extraction for LTCUSDT
######################################################################
Existing rows: 488,475
Existing latest timestamp: 2026-06-06 00:00:00+00:00
Incremental start: 2026-06-05 23:00:00+00:00
End date: 2026-06-12 20:50:57.191570+00:00

LTCUSDT chunk: 2026-06-05 23:00:00+00:00 → 2026-06-12 20:50:57.191570+00:00
Returned 658 rows
Parsed 658 rows from this chunk

New rows before merge: 658

Final shape LTCUSDT: (498191, 6)
Final earliest timestamp: 2025-07-01 18:46:00+00:00
Final latest timestamp: 2026-06-12 17:56:00+00:00
                  timestamp instrument exchange  liquidation_volume  \
0 2025-07-01 18:46:00+00:00    LTCUSDT  binance          1672.80000   
1 2025-07-01 18:47:00+00:00    LTCUSDT  binance           358.71951   
2 2025-07-01 18:48:00+00:00    LTCUSDT  binance             0.00000   
3 2025-07-01 18:49:00+00:00    LTCUSDT  binance             0.00000   
4 2